
# 🥉 Bronze Layer - Raw Ingestion
**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Goal:** Read the raw CSV and register it as a Delta Table with zero transformations.  
This layer is the single source of truth - nothing is modified here.

In [0]:
# Verify that the Spark session and Python environment are ready
# This is the first check we run before any data processing
import sys

try:
    # spark.version accesses the Spark session object that Databricks
    # creates automatically — no need to initialize it manually
    spark_version = spark.version
    
    # sys.version returns the full Python version string
    # we split by space and take the first element to get just the number
    # Example: "3.12.3 (main, ...)" → "3.12.3"
    python_version = sys.version.split(" ")[0]
    
    # Print a clean summary of the environment
    print("=" * 45)
    print("  ENVIRONMENT CHECK")
    print("=" * 45)
    print(f"  Spark version  : {spark_version}")
    print(f"  Python version : {python_version}")
    print("=" * 45)
    print("  Status: cluster is active ✓")
    print("=" * 45)

except Exception as e:
    # If anything above fails, we print a clear error message
    # and re-raise the exception so the notebook stops execution
    # We never want to silently ignore errors in a data pipeline
    print(f"  ✗ Cluster check failed: {e}")
    raise

In [0]:
# Read data directly from the table we created via Data Ingestion
# spark.table() reads from the Databricks catalog using the full table path:
# "catalog.schema.table_name" → "main.default.warehouse_and_retail_sales"
# inferSchema is not needed here because the table already exists in the catalog

df_bronze = spark.table("main.default.warehouse_and_retail_sales")

# Count total rows and columns to verify the data loaded correctly
# f-string with :, formats the number with comma separators → 307,548
print(f"Rows   : {df_bronze.count():,}")
print(f"Columns: {len(df_bronze.columns)}")

# printSchema() shows the data type of each column
# In Bronze everything should be string — we intentionally keep raw types
print("\nSchema:")
df_bronze.printSchema()

## Bronze Layer — Schema Overview

The raw dataset contains **307,645 rows** and **9 columns**.

Databricks automatically inferred the data types during upload:

| Column | Type | Description |
|--------|------|-------------|
| YEAR | long | Year of the transaction (2020–2026) |
| MONTH | long | Month number (1–12) |
| SUPPLIER | string | Distributor name |
| ITEM CODE | string | Product identifier |
| ITEM DESCRIPTION | string | Full product name |
| ITEM TYPE | string | Category: WINE, BEER or LIQUOR |
| RETAIL SALES | double | Units sold at retail level |
| RETAIL TRANSFERS | double | Units transferred between stores |
| WAREHOUSE SALES | double | Units sold from warehouse |

> **Note:** All columns have `nullable = true`, meaning null values
> may exist in any column. This will be handled in the Silver layer.

> **Note:** Since Databricks inferred numeric types correctly on upload,
> the Silver layer will focus on: creating a unified `date` column,
> standardizing text fields, and computing `total_sales`.

In [0]:
# Preview the first 10 rows
# display() is a Databricks-native function — better than .show()
# It renders an interactive table instead of plain text
display(df_bronze.limit(10))